# 5. XGBoost Hyperparameter Tuning

This notebook improves the XGBoost model developed during model comparison.

The main objectives are:

- Load the prepared training and testing datasets
- Handle class imbalance using `scale_pos_weight`
- Build an XGBoost preprocessing and modelling pipeline
- Perform hyperparameter tuning using RandomizedSearchCV
- Use cross-validation on the training data
- Evaluate the tuned model on the untouched test set
- Compare the tuned model with the original XGBoost baseline

The test dataset is not used during hyperparameter selection.

In [5]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier

import joblib

pd.set_option("display.max_columns", None)

#### Load processed data

In [6]:
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")

y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

print("Data loaded successfully.")

print("\nX_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

Data loaded successfully.

X_train: (24000, 39)
X_test: (6000, 39)
y_train: (24000,)
y_test: (6000,)


#### Check target distribution

In [7]:
print("Training target distribution:")
print(y_train.value_counts())

print("\nTraining target proportions:")
print(y_train.value_counts(normalize=True))

Training target distribution:
default payment next month
0    18691
1     5309
Name: count, dtype: int64

Training target proportions:
default payment next month
0    0.778792
1    0.221208
Name: proportion, dtype: float64


#### Calculate class imbalance ratio

In [8]:
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

print("Non-default cases:", negative_count)
print("Default cases:", positive_count)
print("Scale positive weight:", scale_pos_weight)

Non-default cases: 18691
Default cases: 5309
Scale positive weight: 3.520625353173856


#### Define feature groups

In [9]:
categorical_features = [
    "SEX",
    "EDUCATION",
    "MARRIAGE"
]

numerical_features = [
    col for col in X_train.columns
    if col not in categorical_features
]

print("Categorical features:")
print(categorical_features)

print("\nNumber of numerical features:", len(numerical_features))
print("Number of categorical features:", len(categorical_features))
print("Total features:", len(numerical_features) + len(categorical_features))

Categorical features:
['SEX', 'EDUCATION', 'MARRIAGE']

Number of numerical features: 36
Number of categorical features: 3
Total features: 39


#### Create preprocessing pipeline

In [10]:
numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    )
])

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "onehot",
        OneHotEncoder(handle_unknown="ignore")
    )
])

In [11]:
# Create ColumnTransformer

preprocessor = ColumnTransformer([
    (
        "num",
        numeric_pipeline,
        numerical_features
    ),
    (
        "cat",
        categorical_pipeline,
        categorical_features
    )
])

#### Create XGBoost model

In [12]:
xgb_classifier = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,

    scale_pos_weight=scale_pos_weight
)

#### Create complete pipeline

In [13]:
xgb_pipeline = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "classifier",
        xgb_classifier
    )
])

print("XGBoost pipeline created successfully.")

XGBoost pipeline created successfully.


#### Define hyperparameter search space

In [14]:
param_distributions = {
    "classifier__n_estimators": [
        200,
        300,
        500,
        700
    ],

    "classifier__max_depth": [
        3,
        4,
        5,
        6,
        7
    ],

    "classifier__learning_rate": [
        0.02,
        0.05,
        0.08,
        0.10
    ],

    "classifier__min_child_weight": [
        1,
        3,
        5,
        7
    ],

    "classifier__subsample": [
        0.7,
        0.8,
        0.9,
        1.0
    ],

    "classifier__colsample_bytree": [
        0.7,
        0.8,
        0.9,
        1.0
    ],

    "classifier__gamma": [
        0,
        0.1,
        0.3,
        0.5
    ],

    "classifier__reg_alpha": [
        0,
        0.01,
        0.1,
        1
    ],

    "classifier__reg_lambda": [
        1,
        2,
        5,
        10
    ]
}

#### Create cross-validation strategy

In [15]:
cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

#### Create RandomizedSearchCV

In [16]:
random_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_distributions,

    n_iter=30,

    scoring="average_precision",

    cv=cv_strategy,

    verbose=2,

    random_state=42,

    n_jobs=-1,

    return_train_score=True
)

####  Tunning

In [ ]:
print("Starting XGBoost hyperparameter tuning...")
print("This may take some time.")

random_search.fit(
    X_train,
    y_train
)

print("\nHyperparameter tuning completed.")

Starting XGBoost hyperparameter tuning...
This may take some time.
Fitting 5 folds for each of 30 candidates, totalling 150 fits
